# Apache Hudi - Setup & Basic CRUD

Apache Hudi is a data lake framework that enables incremental data processing, upserts, and efficient storage management on top of distributed storage systems.
Unlike traditional Parquet tables, Hudi introduces record-level updates, indexing, and timeline-based versioning, making it suitable for real-time data lakes.

# What you will learn
In this notebook, you will learn:

- What Apache Hudi is and when to use it  
- Difference between Copy-On-Write (COW) and Merge-On-Read (MOR) tables  
- Creating a basic Hudi table using Spark  
- Key concepts: record key, partition path, and precombine field  
- Writing initial data (insert) into a Hudi table  
- Reading data using `format("hudi")`  
- Exploring Hudi metadata columns (`_hoodie_*`)  
- Verifying table structure and stored data  

> Important: the Hudi table path must be on a shared volume that is visible to the Spark driver and all Spark workers.


In [1]:
import os
import shutil
import pathlib
import datetime as dt

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType


## Step 1 — Spark and Hudi configuration

The notebook is prepared for a Docker-based Spark standalone cluster.

- Spark master: `spark://spark-master:7077`
- Shared data folder: `/workspace/data`
- Hudi table path: `/workspace/data/hudi/riders_cow`

In [2]:
MASTER = "spark://spark-master:7077"
WAREHOUSE_DIR = "/workspace/data/warehouse"
HUDI_BASE_DIR = "/workspace/data/hudi"
TABLE_NAME = "riders_cow"
TABLE_PATH = f"{HUDI_BASE_DIR}/{TABLE_NAME}"

pathlib.Path(HUDI_BASE_DIR).mkdir(parents=True, exist_ok=True)
pathlib.Path(WAREHOUSE_DIR).mkdir(parents=True, exist_ok=True)

print(f"Spark master : {MASTER}")
print(f"Table path   : {TABLE_PATH}")


Spark master : spark://spark-master:7077
Table path   : /workspace/data/hudi/riders_cow


## Step 2 — Create SparkSession

Hudi needs the Kryo serializer. The Spark SQL extension and catalog make the session ready for Hudi SQL features as well.


In [3]:
spark = (
    SparkSession.builder
    .appName("Hudi-01-Setup-Basic-CRUD")
    .master(MASTER)
    .config("spark.executor.memory", "2g")
    .config("spark.driver.memory", "1g")
    .config("spark.sql.warehouse.dir", WAREHOUSE_DIR)
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.sql.extensions", "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.hudi.catalog.HoodieCatalog")
    #.config("spark.jars.packages", HUDI_PACKAGE) -- already downloaded in Dockerfile
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Application : {spark.sparkContext.appName}")
print(f"Master      : {spark.sparkContext.master}")


26/04/25 19:19:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
Spark version: 4.0.2
Application : Hudi-01-Setup-Basic-CRUD
Master      : spark://spark-master:7077


## Step 3 — Prepare sample input data

The business columns are:

- `rider` — record key
- `city` — partition path
- `ts` — precombine field

Hudi will automatically add metadata columns such as `_hoodie_commit_time`, `_hoodie_record_key`, and `_hoodie_file_name`.


In [ ]:
schema = StructType([
    StructField("rider", StringType(), False),
    StructField("city", StringType(), False),
    StructField("ts", TimestampType(), False),
])

rows = [
    ("r1", "SFO", dt.datetime(2024, 1, 1, 10, 0, 0)),
    ("r2", "NYC", dt.datetime(2024, 1, 1, 11, 0, 0)),
]

riders_df = spark.createDataFrame(rows, schema)

print(f"Rows prepared: {riders_df.count()}")
riders_df.printSchema()
riders_df.show(truncate=False)


[Stage 0:>                                                          (0 + 0) / 2]

26/04/25 19:19:58 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/04/25 19:20:13 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/04/25 19:20:28 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/04/25 19:20:43 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


[Stage 0:>                                                          (0 + 0) / 2]

26/04/25 19:20:58 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/04/25 19:21:13 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources


## Step 4 — Clean the demo table path

This cell makes the notebook repeatable. It removes the previous demo output before writing the table again.


In [ ]:
if os.path.exists(TABLE_PATH):
    shutil.rmtree(TABLE_PATH)
    print(f"Removed existing table path: {TABLE_PATH}")
else:
    print(f"No existing table path found: {TABLE_PATH}")


## Step 5 — Insert records into a Hudi COPY_ON_WRITE table

This write creates the Hudi table and inserts the two sample riders.

Key options:

- `hoodie.table.type = COPY_ON_WRITE`
- `hoodie.datasource.write.recordkey.field = rider`
- `hoodie.datasource.write.partitionpath.field = city`
- `hoodie.datasource.write.precombine.field = ts`


In [ ]:
hudi_options = {
    "hoodie.table.name": TABLE_NAME,
    "hoodie.datasource.write.table.name": TABLE_NAME,
    "hoodie.datasource.write.operation": "insert",
    "hoodie.datasource.write.recordkey.field": "rider",
    "hoodie.datasource.write.partitionpath.field": "city",
    "hoodie.datasource.write.precombine.field": "ts",
    "hoodie.table.type": "COPY_ON_WRITE",
    "hoodie.datasource.write.hive_style_partitioning": "true",
}

(
    riders_df.write
    .format("hudi")
    .options(**hudi_options)
    .mode("overwrite")
    .save(TABLE_PATH)
)

print(f"Hudi COW table created: {TABLE_PATH}")


## Step 6 — Read the Hudi table

Reading with `format("hudi")` returns both business columns and Hudi metadata columns.


In [ ]:
hudi_df = spark.read.format("hudi").load(TABLE_PATH)

print(f"Rows read: {hudi_df.count()}")
hudi_df.printSchema()
hudi_df.select(
    "_hoodie_commit_time",
    "_hoodie_record_key",
    "_hoodie_partition_path",
    "_hoodie_file_name",
    "rider",
    "city",
    "ts",
).orderBy("rider").show(truncate=False)


## Step 7 — Register a temporary view and query with Spark SQL


In [ ]:
hudi_df.createOrReplaceTempView(TABLE_NAME)

spark.sql(f"""
    SELECT
        rider,
        city,
        ts,
        _hoodie_commit_time,
        _hoodie_record_key,
        _hoodie_partition_path
    FROM {TABLE_NAME}
    ORDER BY rider
""").show(truncate=False)


## Step 8 — Verify the physical table structure

A partitioned Hudi table contains:

- `.hoodie/` metadata and timeline files
- one folder per partition value
- Parquet data files inside partitions


In [ ]:
for root, dirs, files in os.walk(TABLE_PATH):
    level = root.replace(TABLE_PATH, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for file_name in sorted(files)[:10]:
        print(f"{subindent}{file_name}")
    if level >= 2:
        dirs[:] = []


## Step 9 — Basic validation checks

These checks confirm that:

- the table contains two rows
- the table is partitioned by city
- Hudi metadata columns are present


In [ ]:
expected_metadata_columns = {
    "_hoodie_commit_time",
    "_hoodie_commit_seqno",
    "_hoodie_record_key",
    "_hoodie_partition_path",
    "_hoodie_file_name",
}

actual_columns = set(hudi_df.columns)
missing_columns = expected_metadata_columns - actual_columns
row_count = hudi_df.count()
partition_count = hudi_df.select("city").distinct().count()

assert row_count == 2, f"Expected 2 rows, got {row_count}"
assert partition_count == 2, f"Expected 2 partitions, got {partition_count}"
assert not missing_columns, f"Missing Hudi metadata columns: {missing_columns}"

print("Validation passed.")
print(f"Rows       : {row_count}")
print(f"Partitions : {partition_count}")
print(f"Metadata   : {sorted(expected_metadata_columns)}")


## Step 10 — Optional cleanup

Keep this cell commented out if you want to inspect the generated Hudi table after the notebook finishes.


In [ ]:
# shutil.rmtree(TABLE_PATH)
# print(f"Removed demo table: {TABLE_PATH}")
